# Phase 1 MulT — 6-Emotion Classification on CMU-MOSEI

Notebook này train **MulT** (Multimodal Transformer) trên 6 nhãn cảm xúc của bộ dữ liệu CMU-MOSEI:
- **Cảm xúc**: `happy`, `sad`, `angry`, `surprise`, `disgust`, `fear`.
- **Task**: Multi-label classification (một mẫu có thể chứa nhiều cảm xúc đồng thời).
- **Loss**: BCEWithLogitsLoss (Binary Cross Entropy cho từng đầu ra độc lập).
- **Metrics**: Per-emotion F1 & Accuracy, Mean F1 & Accuracy, Mean MAE.

Mục tiêu: Đạt chất lượng nhận diện đa cảm xúc tốt nhất bằng mô hình MulT Transformer.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
else:
    print('Not running inside Google Colab. Mount step skipped.')

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
import sys
from pathlib import Path

RUNTIME_PROFILE = 'colab'
REPO_SOURCE = 'git'
REPO_URL = 'https://github.com/Kandesfx/Training-Multimodal-Emotion-Analysis.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/BCDA')
REPO_PATH = Path('/content/BCDA')
USE_DRIVE_OUTPUTS = True
RESUME_TRAINING = False
RESUME_CHECKPOINT_TYPE = 'last'
BEST_CHECKPOINT_NAME = 'best_model_mult_emotion.pt'
LAST_CHECKPOINT_NAME = 'last_model_mult_emotion.pt'
USE_GCS = True
GCS_BUCKET = 'mer-data-bucket-kandesfx'
WANDB_ENABLE = True
WANDB_PROJECT = 'bcda-phase1'

if RUNTIME_PROFILE != 'colab':
    raise ValueError('Notebook này hiện được tối ưu cho profile colab.')

if REPO_SOURCE not in {'git', 'drive'}:
    raise ValueError("REPO_SOURCE must be either 'git' or 'drive'.")

if REPO_SOURCE == 'git':
    if not REPO_PATH.exists():
        if '<YOUR_REPO_URL_HERE>' in REPO_URL:
            raise ValueError('Hãy thay REPO_URL bằng URL repo thật trước khi chạy cell này.')
        get_ipython().system(f'git clone {REPO_URL} {REPO_PATH}')
    else:
        print(f'Repo already exists at {REPO_PATH}')
else:
    REPO_PATH = DRIVE_ROOT

%cd {REPO_PATH}
if str(REPO_PATH) not in sys.path:
    sys.path.append(str(REPO_PATH))

!python -m pip install -q --upgrade pip
!python -m pip install -q torch torchvision torchaudio numpy pandas scikit-learn matplotlib seaborn tqdm wandb

if IN_COLAB and USE_GCS:
    from google.colab import auth
    print('Authenticating for GCS access...')
    auth.authenticate_user()
    print('Downloading aligned_50.pkl from GCS...')
    get_ipython().system(f'mkdir -p /content/data/MSA-Dataset')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/data/MSA-Dataset/aligned_50.pkl /content/data/MSA-Dataset/aligned_50.pkl')

In [ ]:
from training.config_phase1 import Phase1Config
from training.dataset_mosei import create_dataloaders
from training.trainer import Phase1Trainer
from training.models.mult import MulTRegressor

# Build config
config = Phase1Config()
config.model_type = 'mult'
config.training.task_type = 'emotion'      # <--- Chuyển sang train emotion
config.mult_model.output_dim = 6            # <--- 6 outputs cho 6 emotions
config.runtime.use_drive_outputs_on_colab = USE_DRIVE_OUTPUTS
config.runtime.use_gcs = USE_GCS
config.runtime.gcs_bucket = GCS_BUCKET
config.wandb.enable = WANDB_ENABLE
config.wandb.project = WANDB_PROJECT
config.apply_profile('colab', drive_root=DRIVE_ROOT, repo_root=REPO_PATH)

# === MulT-specific hyperparameters ===
config.mult_model.d_model = 64
config.mult_model.num_heads = 4
config.mult_model.num_cross_layers = 4
config.mult_model.num_self_layers = 2
config.mult_model.ffn_dim = 128
config.mult_model.attn_dropout = 0.2
config.mult_model.fusion_hidden_dim = 128
config.mult_model.fusion_dropout = 0.5

# === Training hyperparameters ===
config.training.batch_size = 32
config.training.num_workers = 2
config.training.num_epochs = 50
config.training.patience = 10
config.training.scheduler_patience = 4
config.training.learning_rate = 1e-4
config.training.scheduler_type = 'cosine_warmup'
config.training.warmup_epochs = 5
config.training.min_lr = 1e-6
config.training.weight_decay = 5e-3
config.training.max_grad_norm = 0.5
config.training.use_amp = True
config.training.resume_from_checkpoint = RESUME_TRAINING
config.training.resume_checkpoint_type = RESUME_CHECKPOINT_TYPE
config.training.checkpoint_name = BEST_CHECKPOINT_NAME
config.training.last_checkpoint_name = LAST_CHECKPOINT_NAME
config.setup()

if USE_GCS and RESUME_TRAINING:
    print('Checking GCS for existing checkpoints to resume...')
    get_ipython().system(f'mkdir -p {config.paths.checkpoints_dir}')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/checkpoints/phase1/{BEST_CHECKPOINT_NAME} {config.paths.checkpoints_dir}/{BEST_CHECKPOINT_NAME} || true')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/checkpoints/phase1/{LAST_CHECKPOINT_NAME} {config.paths.checkpoints_dir}/{LAST_CHECKPOINT_NAME} || true')
    get_ipython().system(f'mkdir -p {config.paths.logs_dir}')
    get_ipython().system(f'gcloud storage cp gs://{GCS_BUCKET}/logs/phase1/history.csv {config.paths.logs_dir}/history.csv || true')

if WANDB_ENABLE:
    import wandb
    wandb.login()

print('Runtime profile:', config.runtime.profile)
print('Model type:', config.model_type)
print('Task type:', config.training.task_type)
print('Resume training:', config.training.resume_from_checkpoint)
print('Best checkpoint name:', config.training.checkpoint_name)
print()
print('=== MulT Model Config ===')
print(f'd_model: {config.mult_model.d_model}')
print(f'num_heads: {config.mult_model.num_heads}')
print(f'num_cross_layers: {config.mult_model.num_cross_layers}')
print(f'num_self_layers: {config.mult_model.num_self_layers}')
print(f'ffn_dim: {config.mult_model.ffn_dim}')
print(f'attn_dropout: {config.mult_model.attn_dropout}')
print(f'fusion_hidden_dim: {config.mult_model.fusion_hidden_dim}')
print(f'fusion_dropout: {config.mult_model.fusion_dropout}')
print(f'output_dim: {config.mult_model.output_dim}')
print()
print('=== Training Config ===')
print(f'batch_size: {config.training.batch_size}')
print(f'learning_rate: {config.training.learning_rate}')
print(f'weight_decay: {config.training.weight_decay}')
print(f'max_grad_norm: {config.training.max_grad_norm}')
print(f'patience: {config.training.patience}')
for key, value in config.paths.as_dict().items():
    print(f'{key}: {value}')

In [ ]:
required_paths = [
    config.paths.mosei_pkl,
    config.paths.checkpoints_dir,
    config.paths.logs_dir,
    config.paths.outputs_dir,
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))

best_checkpoint_path = config.paths.checkpoints_dir / config.training.checkpoint_name
last_checkpoint_path = config.paths.checkpoints_dir / config.training.last_checkpoint_name

print(f'Aligned pkl: {config.paths.mosei_pkl}')
print(f'Best checkpoint path: {best_checkpoint_path}')
print(f'Last checkpoint path: {last_checkpoint_path}')
print(f'Best checkpoint exists: {best_checkpoint_path.exists()}')
print(f'Last checkpoint exists: {last_checkpoint_path.exists()}')
print('All required paths are ready.')

In [ ]:
# === Smoke test: MulTRegressor with Emotion Config ===
import torch

print('=== Smoke Test: MulTRegressor (Emotion Mode) ===')
model = MulTRegressor(config.mult_model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

B, T = 4, 50
text = torch.randn(B, T, 768)
audio = torch.randn(B, T, 74)
vision = torch.randn(B, T, 35)

# Forward pass
output = model(text=text, audio=audio, vision=vision)
assert output.shape == (B, 6), f'Expected shape ({B}, 6), got {output.shape}'
print(f'Forward pass OK (output_dim=6). Output shape: {output.shape}')

# Backward pass
loss = output.sum()
loss.backward()
print('Backward pass OK.')
print('All smoke tests passed!')

In [ ]:
# === Load emotion-enabled dataloaders ===
dataloaders = create_dataloaders(config=config, pkl_path=config.paths.mosei_pkl)

sample_batch = next(iter(dataloaders['train']))
print('=== Sample Batch Keys & Shapes ===')
for key, value in sample_batch.items():
    if hasattr(value, 'shape'):
        print(f'  {key}: {value.shape}  dtype={value.dtype}')
    else:
        print(f'  {key}: type={type(value).__name__}')

print()
print('Dataset sizes:')
for split, loader in dataloaders.items():
    print(f'  {split}: {len(loader.dataset)} samples, {len(loader)} batches')

# Inspect labels to confirm they are (B, 6) emotion vectors
label_batch = sample_batch['label']
assert label_batch.shape == (config.training.batch_size, 6), f'Unexpected label shape: {label_batch.shape}'
print('Confirmed: Labels loaded as emotion vectors with shape (B, 6)')

EMOTIONS = ['happy', 'sad', 'angry', 'surprise', 'disgust', 'fear']
for i, name in enumerate(EMOTIONS):
    col = label_batch[:, i].numpy()
    n_present = (col >= 0.5).sum()
    print(f'  {name:>10}: {n_present:>2}/{len(col)} present (intensity >= 0.5) in sample batch')

In [ ]:
# === Instantiate model and start training ===
model = MulTRegressor(config.mult_model)
trainer = Phase1Trainer(model=model, config=config)
summary = trainer.fit(dataloaders['train'], dataloaders['valid'])
test_metrics = trainer.evaluate_and_save(dataloaders['test'], split='test', epoch=summary['best_epoch'])
summary, test_metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_path = config.paths.logs_dir / 'history.csv'
history_df = pd.read_csv(history_path)
history_df.tail(20)

In [ ]:
valid_df = history_df[history_df['split'] == 'valid'].copy()
train_df = history_df[history_df['split'] == 'train'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Loss curves
axes[0].plot(train_df['epoch'], train_df['loss'], label='train_loss', marker='o', markersize=3)
axes[0].plot(valid_df['epoch'], valid_df['loss'], label='valid_loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (BCE)')
axes[0].set_title('BCE Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Mean F1 and Mean Accuracy
axes[1].plot(valid_df['epoch'], valid_df['mean_f1'], label='valid_mean_f1', marker='o', markersize=3, color='tab:orange')
axes[1].plot(valid_df['epoch'], valid_df['mean_acc'], label='valid_mean_acc', marker='s', markersize=3, color='tab:green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Metric Value')
axes[1].set_title('Validation Aggregate Metrics')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 3. Per-emotion F1 curves
plt.figure(figsize=(10, 6))
for emo in EMOTIONS:
    col = f'{emo}_f1'
    if col in valid_df.columns:
        plt.plot(valid_df['epoch'], valid_df[col], label=f'{emo} F1', marker='o', markersize=3)
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.title('Validation Per-Emotion F1 Scores')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# === Test Set Performance ===
test_row = history_df[history_df['split'] == 'test'].tail(1)
if len(test_row) > 0:
    print('=== Final Test Set Emotion Performance ===')
    print(f'Mean F1:       {test_row["mean_f1"].values[0]:.4f}')
    print(f'Mean Accuracy: {test_row["mean_acc"].values[0]:.4f}')
    print(f'Mean MAE:      {test_row["mean_mae"].values[0]:.4f}')
    print()
    print('Per-Emotion F1 Scores:')
    for emo in EMOTIONS:
        col = f'{emo}_f1'
        if col in test_row.columns:
            print(f'  {emo:<10}: {test_row[col].values[0]:.4f}')